In [ ]:
print("Supervised Modeling Notebook")

In [ ]:
# Load Cleaned Data
import pandas as pd

df = pd.read_csv("../data/cleaned_data.csv")

df.head()

In [ ]:
# Define Features & Target (Regression)

X = df.drop(columns=["price"])
y = df["price"]

In [ ]:
# Identify Column Types

categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Create Preprocessing Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [ ]:
# Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Build Regression Model (SGD)

from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SGDRegressor(
        learning_rate='constant',
        eta0=0.01,
        max_iter=1000
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
# Train Model

model.fit(X_train, y_train)

In [ ]:
# Predict

y_pred = model.predict(X_test)

In [ ]:
# Evaluate Model

from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)

In [ ]:
# Cross Validation

from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')

print("Average MSE:", -scores.mean())

In [ ]:
# EXPERIMENT part

models = [
    SGDRegressor(eta0=0.001, max_iter=500),
    SGDRegressor(eta0=0.01, max_iter=1000),
    SGDRegressor(eta0=0.1, max_iter=2000)
]

for m in models:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", m)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    print("Model:", m)
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("------")

In [ ]:
# Visualization

import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")
plt.show()

* Learning Rate

Learning rate controls how fast the model updates its weights.

A small learning rate results in slow but stable learning.
A large learning rate results in faster learning but may cause unstable behavior or overshooting.


* Epochs

Epochs represent how many times the model sees the entire dataset.

Too few epochs → underfitting (model does not learn enough).
Too many epochs → overfitting (model memorizes training data).


* Batch Size

Batch size defines how many samples are used before updating the model.

Small batch size → faster but noisy updates.
Large batch size → more stable but slower learning.


* MSE vs MAE

Mean Squared Error (MSE) squares the errors, so large errors are penalized more.

Mean Absolute Error (MAE) calculates the average absolute difference.

MSE is sensitive to outliers, while MAE gives a more general idea of error.


* Underfitting vs Overfitting

Underfitting occurs when the model performs poorly on both training and testing data.

Overfitting occurs when the model performs well on training data but poorly on testing data.

Cross-validation helps detect these issues.


* Business Impact

If the model predicts price too high, customers may not purchase the product.

If the model predicts price too low, the company may lose profit.

Large prediction errors (high MSE) can lead to incorrect pricing strategies and financial loss.

Therefore, minimizing prediction error is critical for business success.

